# Clean CM Operator Data Reports
Code author: Audrey McManemin

Date started: 2024-10-18
Date last edited: 2024-10-18


### Notes
- All raw report inputs are saved in 00_raw_reports. No changes are manually made to operator reports (except for date propagation in release schedule). All cleaning is handled in Python.


In [6]:
# Imports
import pandas as pd
import numpy as np
import pathlib
from methods_source_cm import RELEASE_NUMBER_LIST

# Load and clean raw data submitted by operators
### Notes on formatting:
- Operators added their own QC indicators, thus not all columns are uniform across reports
- Values left in the Excel file are replaced during import into PyCharm with "nan"
- Naming convention for dataframes: operator

## Notes on Cleaning Operator Data

### Generate data frame with the following columns:
- Operator: name of operator (Sensirion, Sensia, SLB)
- Week: week(s) that the operator participated in (1, 2, 3, 4)
- ReleaseID: release number for that week, corresponding to the master schedule of all the releases (not just for which that operator measured)
- DateOfSurvey: date in YYYY-MM-DD format
- EmissionStartTime: start time of survey in local time (UTC+2)
- EmissionEndTime: end time of survey in local time (UTC+2)
- QuantifiedPlume: boolean input, 1 indicates operator submitted a valid quantification estimate for this overpass (excludes quantification estimates that are provided but fail operator QC standards)
- EstimatedEmissionRate: estimated emissions in kgh
- EstimatedEmissionRateUpper: upper bound of uncertainty on quantification estimate
- EstimatedEmissionRateLower: lower bound of uncertainty on quantification estimate
- UncertaintyType: type of uncertainty for upper and lower values reported above
- OperatorWindspeed: operator reported windspeed in m/s
- QCFlag: operator specific QC flag
- OperatorKept: operator submitted estimates for this result
- EstimateType: if applicable, the method used for this estimate
- EstimatedSourceLatitude: the estimated latitude of the methane source
- EstimatedSourceLongitude: the estimated longitude of the methane source
- Alarm: indicates (Y/N) whether an alarm notification would be sent to a customer based on this and/or previous detections.


In [7]:
# %% Stanford QC analysis: depends on the explanation
def stanford_qc(release, schedule):
    measurement_taken = schedule.loc[release-1, "Measurement Taken"].lower()
    quantification_status = schedule.loc[release-1, "Quantification Status"].lower()
    
    if measurement_taken == 'yes' and quantification_status == 'completed':
        return True
    elif measurement_taken == 'yes' and quantification_status == 'failed':
        # slb releases below LOD
        if schedule.loc[release-1, "Explanation"] == "below LOD or obscured":
            return True
        # sensirion release below LOD
        elif (schedule.loc[release-1, "Explanation"] in ("Low signals measured on sensors")) or ("no signals" in schedule.loc[release-1, "Explanation"]):
            return True
        else: 
            return False
    
    return False 

In [8]:
# Operator QC analysis: only counted if quantification was completed
def operator_qc(measurement_taken, quantification_status):
    if measurement_taken == 'no':
        return False
    elif (measurement_taken == 'yes') and (quantification_status == 'failed'):
        return False
    elif (measurement_taken == 'yes') and (quantification_status == 'completed'):
        return True 

    return np.nan

In [9]:
# Strict QC: anything that failed quantification is counted as a zero 
def strict_qc(measurement_taken, quantification_status):
    if measurement_taken == 'no':
        return False
    elif measurement_taken == 'yes':
        return True 
    
    return np.nan

### Sensirion
#### Submission details
- Participated in all 4 weeks of the experiment
- Submitted all results on time

#### Required data cleaning
- Missing data reporting: system was offline during the first 5 releases of Day 1, Week 1 (2024-06-07) due to installation
- Sensirion included all emissions, even ones they did not detect, in their results.

In [16]:
# %% Sensirion data cleaning

def clean_sensirion(results, schedule, week, discard_level=None):
    operator = 'Sensirion'
    num_releases = range(1, RELEASE_NUMBER_LIST[week] + 1) # for loop index
    release_list = [] # generating all new rows
    
    schedule["Explanation"] = schedule["Explanation"].fillna('')
    
    for release in num_releases:
        
        if pd.notna(results.loc[release-1, "EstimatedEmissionRate"]):
            quantified = True
            emission_rate = results.loc[release-1, 'EstimatedEmissionRate']
            emission_upper = results.loc[release-1, 'EstimatedEmissionRateUpper']
            emission_lower = results.loc[release-1, 'EstimatedEmissionRateLower']
            QCflag = ''
        # only offline time is first 5 releases in W1 - not operator kept
        elif (week == 1 and release in [1, 2, 3, 4, 5]):
            quantified = False
            emission_rate = np.nan
            emission_upper = np.nan
            emission_lower = np.nan
            QCflag = schedule.loc[release-1, "Explanation"]

        # make everything else zeros
        else:
            quantified = True
            emission_rate = 0
            emission_upper = 0
            emission_lower = 0
            QCflag = schedule.loc[release-1, "Explanation"]
        
        start_time = results.loc[release-1, "EmissionStartTime"]
        end_time = results.loc[release-1, "EmissionEndTime"]
        
        ## QC analysis
        measurement_taken = schedule.loc[release-1, "Measurement Taken"].lower()
        quantification_status = schedule.loc[release-1, "Quantification Status"].lower()
        
        operator_keep = operator_qc(measurement_taken, quantification_status)
        stanford_keep = stanford_qc(release, schedule)
        strict_qc_keep = strict_qc(measurement_taken, quantification_status)
        
        # specifically for issue in Week 2 where "no signals" are reported as NO measurement taken
        if ("no signals" in schedule.loc[release-1, "Explanation"]):
            strict_qc_keep = True
            stanford_keep = True
        
        new_row = {
            'Operator': operator,
            'Week': week,
            'DateOfSurvey': results.loc[release-1, "DateOfSurvey"],
            'ReleaseID': release, 
            'SurveyStartTime': start_time,
            'SurveyEndTime': end_time,
            'QuantifiedPlume': quantified,
            'EstimatedEmissionRate': emission_rate,
            'EstimatedEmissionRateUpper': emission_upper,
            'EstimatedEmissionRateLower': emission_lower,
            'UncertaintyType': results.loc[release-1, 'UncertaintyType'],
            'OperatorWindspeed': results.loc[release-1, "WindSpeed"],
            'QCFLag': QCflag,
            'OperatorKeep': operator_keep,
            'StanfordKeep': stanford_keep,
            'StrictQCKeep': strict_qc_keep,
        }
        
        release_list.append(new_row)

    clean_df = pd.DataFrame(release_list)
    
    return clean_df


In [17]:
# import data

sensirion_clean = pd.DataFrame()
df_list = []
for week in [1, 2, 3, 4]:
    sensirion_results_path = pathlib.PurePath('00_raw_reports', f'Sensirion_results_W{week}.xlsx')
    sensirion_results = pd.read_excel(sensirion_results_path, sheet_name='Reported Data', engine='openpyxl')
    sensirion_schedule_path = pathlib.PurePath('00_raw_reports', f'Sensirion_schedule_W{week}.xlsx')
    sensirion_schedule = pd.read_excel(sensirion_schedule_path, engine='openpyxl', skiprows=1, usecols='D:J')
    
    # clean data
    df = clean_sensirion(sensirion_results, sensirion_schedule, week)
    df_list.append(df)

sensirion_clean = pd.concat(df_list)
sensirion_clean.rename(columns={'ReleaseID': 'WeeklyReleaseID'}, inplace=True)
sensirion_clean = sensirion_clean.reset_index()
sensirion_clean['ReleaseID'] = sensirion_clean.index + 1
sensirion_clean.shape

# save data
sensirion_clean.to_csv(pathlib.PurePath('01_clean_reports', 'Sensirion_clean.csv'), index=False)


In [143]:
print(f"Number of releases: {sensirion_clean.shape[0]}")
sensirion_clean.head(10)

Number of releases: 152


,index,Operator,Week,DateOfSurvey,WeeklyReleaseID,SurveyStartTime,SurveyEndTime,QuantifiedPlume,EstimatedEmissionRate,EstimatedEmissionRateUpper,EstimatedEmissionRateLower,UncertaintyType,OperatorWindspeed,QCFLag,OperatorKeep,ReleaseID
0,0,Sensirion,1,17-06-2024,1,11:07:00,11:53:00,False,NaN,NaN,NaN,NaN,NaN,Installation not yet finished,False,1
1,1,Sensirion,1,17-06-2024,2,12:05:00,12:51:00,False,NaN,NaN,NaN,NaN,NaN,Installation not yet finished,False,2
2,2,Sensirion,1,17-06-2024,3,13:05:00,13:59:00,False,NaN,NaN,NaN,NaN,NaN,Installation not yet finished,False,3
3,3,Sensirion,1,17-06-2024,4,14:00:00,14:48:00,False,NaN,NaN,NaN,NaN,NaN,Installation not yet finished,False,4
4,4,Sensirion,1,17-06-2024,5,15:00:00,15:46:00,False,NaN,NaN,NaN,NaN,NaN,Installation not yet finished,False,5
5,5,Sensirion,1,17-06-2024,6,16:00:00,16:47:00,True,0.95,3.60,0.41,95% Cl,2.048584,,True,6
6,6,Sensirion,1,17-06-2024,7,16:57:00,17:58:00,True,133.86,300.00,0.34,95% Cl,2.473270,,True,7
7,7,Sensirion,1,18-06-2024,8,09:08:00,09:54:00,True,51.51,51.51,7.70,95% Cl,1.477687,,True,8
8,8,Sensirion,1,18-06-2024,9,10:05:00,10:51:00,True,24.57,24.57,3.40,95% Cl,0.993440,,True,9
9,9,Sensirion,1,18-06-2024,10,11:03:00,11:49:00,True,3.60,3.60,0.52,95% Cl,1.303866,,True,10


### Sensia
#### Submission details
- Participated in weeks 3 and 4 of the experiment
- Submitted all results on time

#### Required data cleaning
- Only submitted results (no schedule)
- Missing data reporting: system was offline during the first 2 releases of Day 1, Week 3 (2024-09-09) due to installation
- Sensia included all emissions minus the offline time (but including undetected releases as zeros) in their results

In [19]:
# %% Sensia data cleaning

def clean_sensia(results):
    operator = 'Sensia'
    total_releases = 0
    for week in [3, 4]:
        total_releases += RELEASE_NUMBER_LIST[week]
    num_releases = range(1, total_releases + 1) # for loop index
    release_list = [] # generating all new rows
    
    result_counter = 1
    for release in num_releases:
         # offline time in first 2 releases Week 3 and last 4 releases Week 3 - not operator kept
        if (release in [1, 2, 74, 75, 76, 77]):
            quantified = np.nan
            start_time = ''
            end_time = ''
            emission_rate = np.nan
            emission_upper = np.nan 
            emission_lower = np.nan 
            QCflag = ''
            uncertainty_type = ''
            operator_windspeed = ''
            operator_keep = False
            stanford_keep = False
            strict_qc_keep = False
            
        elif pd.notna(results.loc[result_counter-1, "EstimatedEmissionRate"]):
                 
            quantified = True
            start_time = results.loc[result_counter-1, "EmissionStartTime"]
            end_time = results.loc[result_counter-1, "EmissionEndTime"]
            emission_rate = results.loc[result_counter-1, 'EstimatedEmissionRate']
            emission_upper = results.loc[result_counter-1, 'EstimatedEmissionRateUpper']
            emission_lower = results.loc[result_counter-1, 'EstimatedEmissionRateLower']
            QCflag = results.loc[result_counter-1, "Comments"]
            operator_keep = True
            uncertainty_type = results.loc[result_counter-1, 'UncertaintyType']
            operator_windspeed = results.loc[result_counter-1, "WindSpeed"]
            operator_keep = True
            strict_qc_keep = True
            stanford_keep = True
            
            # increment results counter
            result_counter = result_counter + 1
            
        
        if release <= RELEASE_NUMBER_LIST[3]:
            week = 3
            weekly_release_id = release
        else:
            week = 4
            weekly_release_id = release - RELEASE_NUMBER_LIST[3]
        
        new_row = {
            'Operator': operator,
            'Week': week,
            'DateOfSurvey': results.loc[release-1, "DateOfSurvey"],
            'ReleaseID': release, 
            'WeeklyReleaseID': weekly_release_id,
            'SurveyStartTime': start_time,
            'SurveyEndTime': end_time,
            'QuantifiedPlume': quantified,
            'EstimatedEmissionRate': emission_rate,
            'EstimatedEmissionRateUpper': emission_upper,
            'EstimatedEmissionRateLower': emission_lower,
            'UncertaintyType': uncertainty_type,
            'OperatorWindspeed': operator_windspeed,
            'QCFLag': QCflag,
            'OperatorKeep': operator_qc,
            'OperatorKeep': operator_keep,
            'StanfordKeep': stanford_keep,
            'StrictQCKeep': strict_qc_keep,
        }
        
        release_list.append(new_row)

    clean_df = pd.DataFrame(release_list)
    
    return clean_df

In [20]:
# import data
sensia_results_path = pathlib.PurePath('00_raw_reports', f'Sensia_results.xlsx')
sensia_results = pd.read_excel(sensia_results_path, sheet_name='Reported Data', engine='openpyxl')

# clean data
sensia_clean = clean_sensia(sensia_results)

# save data
sensia_clean.to_csv(pathlib.PurePath('01_clean_reports', 'Sensia_clean.csv'), index=False)

In [146]:
print(f"Number of releases: {sensia_clean.shape[0]}")
sensia_clean.head(10)

Number of releases: 77


,Operator,Week,DateOfSurvey,ReleaseID,WeeklyReleaseID,SurveyStartTime,SurveyEndTime,QuantifiedPlume,EstimatedEmissionRate,EstimatedEmissionRateUpper,EstimatedEmissionRateLower,UncertaintyType,OperatorWindspeed,QCFLag,OperatorKeep
0,Sensia,3,2024-09-09,1,1,,,False,NaN,NaN,NaN,,,,False
1,Sensia,3,2024-09-09,2,2,,,False,NaN,NaN,NaN,,,,False
2,Sensia,3,2024-09-09,3,3,13:30:00,13:45:00,True,3.200,4.932,1.468,"""+/-""",NaN,NaN,True
3,Sensia,3,2024-09-09,4,4,14:37:00,15:15:00,True,0.000,0.000,0.000,-,NaN,NaN,True
4,Sensia,3,2024-09-10,5,5,15:25:00,16:15:00,True,2.600,4.358,0.842,"""+/-""",NaN,NaN,True
5,Sensia,3,2024-09-10,6,6,16:42:00,17:37:00,True,8.203,11.167,5.239,"""+/-""",NaN,NaN,True
6,Sensia,3,2024-09-10,7,7,08:45:00,09:31:00,True,1.554,2.548,0.560,"""+/-""",NaN,NaN,True
7,Sensia,3,2024-09-10,8,8,09:45:00,10:31:00,True,5.879,8.888,2.870,"""+/-""",NaN,NaN,True
8,Sensia,3,2024-09-10,9,9,10:40:00,11:26:00,True,63.102,79.163,47.041,"""+/-""",NaN,NaN,True
9,Sensia,3,2024-09-10,10,10,11:45:00,12:31:00,True,47.990,68.022,27.958,"""+/-""",NaN,NaN,True


### SLB
#### Submission details
- Participated in weeks 3 and 4 of the experiment
- Submitted all results on time

#### Required data cleaning
- Submitted combined results and then separate schedules for W3 and W4.
- Missing data reporting: system was offline during the 4th and 5th releases of Day 1, Week 3 (2024-09-09)
- SLB only included completed estimates in their report
- Measurement failed for those with "below LOD or obscured"

In [16]:
# %% SLB data cleaning

def clean_slb(results, schedule):
    operator = 'SLB'
    num_releases = range(1, RELEASE_NUMBER_LIST[3] + RELEASE_NUMBER_LIST[4] + 1) # for loop index
    release_list = [] # generating all new rows
    
     # fill na with 'None' for quantification status
    schedule['Quantification Status'] = schedule['Quantification Status'].fillna('None')
    schedule['Measurement Taken'] = schedule['Measurement Taken'].fillna('None')
    
    release_estimate_index = 1
    
    for release in num_releases:
        
        if schedule.loc[release-1, "Quantification Status"] == 'Completed':
            quantified = True
            start_time = results.loc[release_estimate_index - 1, "EmissionStartTime"]
            end_time = results.loc[release_estimate_index - 1, "EmissionEndTime"]
            emission_rate = results.loc[release_estimate_index - 1, 'EstimatedEmissionRate']
            emission_upper = results.loc[release_estimate_index - 1, 'EstimatedEmissionRateUpper']
            emission_lower = results.loc[release_estimate_index - 1, 'EstimatedEmissionRateLower']
            uncertainty_type = results.loc[release_estimate_index - 1, 'EstimateType']
            windspeed = results.loc[release_estimate_index, "WindSpeed"]
            release_estimate_index += 1
        
        elif schedule.loc[release-1, "Measurement Taken"] == 'YES' and schedule.loc[release-1, "Quantification Status"] == 'Failed':
            # they didn't actually quantify but could be because it was below LOD
            quantified = True
            start_time = schedule.loc[release-1, "Start Time"]
            end_time = schedule.loc[release-1, "End Time"]
            emission_rate = 0.0
            emission_upper = 0.0
            emission_lower = 0.0
            uncertainty_type = ''
            windspeed = np.nan
            
        else:
            quantified = False
            start_time = np.nan
            end_time = np.nan
            emission_rate = np.nan 
            emission_upper = np.nan
            emission_lower = np.nan 
            uncertainty_type = '' 
            windspeed = np.nan 
        
        if release <= RELEASE_NUMBER_LIST[3]:
            week = 3
            weekly_release_id = release
        else:
            week = 4
            weekly_release_id = release - RELEASE_NUMBER_LIST[3]
        
        ## QC analysis
        measurement_taken = schedule.loc[release-1, "Measurement Taken"].lower()
        quantification_status = schedule.loc[release-1, "Quantification Status"].lower()
        
        operator_keep = operator_qc(measurement_taken, quantification_status)
        stanford_keep = stanford_qc(release, schedule)
        strict_qc_keep = strict_qc(measurement_taken, quantification_status)
        
            
        new_row = {
            'Operator': operator,
            'Week': week,
            'DateOfSurvey': schedule.loc[release-1, "Date"],
            'ReleaseID': release, 
            'SurveyStartTime': start_time,
            'SurveyEndTime': end_time,
            'QuantifiedPlume': quantified,
            'EstimatedEmissionRate': emission_rate,
            'EstimatedEmissionRateUpper': emission_upper,
            'EstimatedEmissionRateLower': emission_lower,
            'UncertaintyType': uncertainty_type,
            'OperatorWindspeed': windspeed,
            'QCFLag': schedule.loc[release-1, "Explanation"],
            'OperatorKeep': operator_keep,
            'StanfordKeep': stanford_keep,
            'StrictQCKeep': strict_qc_keep,
        }
            
        release_list.append(new_row)

    clean_df = pd.DataFrame(release_list)
    
    return clean_df


In [17]:
# import data
slb_results_path = pathlib.PurePath('00_raw_reports', f'SLB_results.xlsx')
slb_schedule_path_W3 = pathlib.PurePath('00_raw_reports', f'SLB_schedule_W3.xlsx')
slb_schedule_path_W4 = pathlib.PurePath('00_raw_reports', f'SLB_schedule_W4.xlsx')

# read data
slb_results = pd.read_excel(slb_results_path, sheet_name='Reported Data', engine='openpyxl')
slb_schedule_W3 = pd.read_excel(slb_schedule_path_W3, engine='openpyxl', skiprows=1, usecols='D:J')
slb_schedule_W3 = slb_schedule_W3[0: RELEASE_NUMBER_LIST[3]]
slb_schedule_W4 = pd.read_excel(slb_schedule_path_W4, engine='openpyxl', skiprows=1, usecols='D:J')
slb_schedule_W4 = slb_schedule_W4[0: RELEASE_NUMBER_LIST[4]]
slb_schedule = pd.concat([slb_schedule_W3, slb_schedule_W4])
slb_schedule = slb_schedule.reset_index()

# clean data
slb_clean = clean_slb(slb_results, slb_schedule)

# save data
slb_clean.to_csv(pathlib.PurePath('01_clean_reports', 'SLB_clean.csv'), index=False)

In [10]:
slb_schedule.loc[35:40, :]

,index,Date,Release Number,Start Time,End Time,Measurement Taken,Quantification Status,Explanation
35,35,2024-09-13 00:00:00,4.0,11:45:00,12:33:00,YES,Completed,NaN
36,36,2024-09-13 00:00:00,5.0,12:45:00,13:34:00,YES,Completed,NaN
37,37,NaT,NaN,None,None,None,None,NaN
38,38,NaT,NaN,None,None,None,None,NaN
39,0,Monday 9/16,1.0,11:00:00,11:46:00,YES,Failed,below LOD or obscured
40,1,Monday 9/16,2.0,11:59:00,12:47:00,YES,Completed,NaN
